# Introduction

The model that I will train on this dataset is the torchgeo's pretrained `resnet-18` encoder model, pretrained on `224x224` `sentinel-2` images having all `13` bands, combined with a regular non-pretrained `UNet` decoder.

The new model requires its own preprocessing step: dividing all `13` bands by 10,000 and then clipping to 0-1 range

Our dataset intentionally has `60m` resolution bands `(bands 1, 9, and 10)` removed.
To make our data match the input expected by the model, we need to add `3` filler bands.

Expected bands:

`B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12`

Bands used:

`B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12`

To solve this problem, filler bands will be added, containing all zeros, for compatibility with the model.

The standard input resolution for `resnet-18` models is `224x224,` however that resolution is too large for our dataset as `5` of the `9` cities' images have atleast one dimension smaller than `224`. Furthermore, if `224x224` patch size is used, a large amount of padding will be needed to be applied, especially for `Kianda` with size `42x34`. As a result, I have decided to make `56x56` patches and scale them by `4x` to `224x224` during training, to preserve the resolution of images passing through the model.

Many types of padding can be applied, however reflect padding is particularly useful for the following reasons:
1. Does not produce artificial pixels, like in zero/constant padding.
2. Avoids sharp transitions and artificial edges.
3. Preserves local spatial statistics, spectral patterns, and textures near the boundary. This is particularly important since `CNN` models operate heavily on local neighbourhoods.

Adding padding, splitting the images into `56x56` patches, and then dividing those patches into `train/val/test` splits is particularly problematic since it needs to ensured that areas that are reflected in the padding appear in the same set: `train`, `validation` or `test`, and are not split between them. For example, a patch near the original image edge might appear inside the `train` set, and its reflected version in the padded image might appear in the `test` set, meaning that the `train-test` unseen data rule has been broken. To avoid this issue I devised this strategy:

Any pixels that were reflected in the padding and all padded pixels are marked as `reflection pixels`. Patches containing `Reflection pixels` will only be included in the `training` set and never in the `test/validation` set to prevent duplicate pixels appearing in different sets. Patches not containing any `reflection pixel` will be split into `train-val-test` sets ensuring that the total distribution of patches is `70%:15%:15%` for `train:val:test`.

Due to the small dimensions of some of the cites, splitting them individually into reasonably `train-val-test` patches is impossible. For example, `Kianda's` image is of shape `42x34` meaning that only one patch will be produced, and thus a `train-val-test` split cannot be done. Thus, only a global `train-val-test` set will be created.

Stratified splitting will be done to preserve spatial distribution of classes in the sets. Stratification labels will be assigned based on how much % of the mask is covered by `class-1 (informal settlement)` pixels. Specifically, I will use `4` bins: `0% - 25%`, `25% - 50%`, `50% - 75%`, `75% - 100%`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from slum_segmentation import helper_functions, np_dataset
from sklearn.model_selection import train_test_split
import gc

Combine image and mask, then pad the combined image and then split into `56x56` patches, then add 3 filler bands for model compatibility, then concatenate all at the end

In [3]:
reflection_patches_list = []
other_patches_list = []

for city in vars(np_dataset.cities).values():
    print(city.name)
    image = city.get_image()
    mask = city.get_mask()

    # model compatibility preprocessing
    image = image / 10000

    city_patch = np.concatenate([image, mask[np.newaxis, :, :]], axis=0)
    city_reflection_patches, city_other_patches = helper_functions.image_to_patches(city_patch, 56)
    city_reflection_patches, city_other_patches = (
        helper_functions.add_filler_bands(city_reflection_patches),
        helper_functions.add_filler_bands(city_other_patches)
    )
    reflection_patches_list.append(city_reflection_patches)
    other_patches_list.append(city_other_patches)

    print("city patch:", city_patch.shape)
    print("city reflection patches:", city_reflection_patches.shape)
    print("city other patches:", city_other_patches.shape)
    print("-"*50)

reflection_patches = np.concatenate(reflection_patches_list, axis=0)
other_patches = np.concatenate(other_patches_list, axis=0)

print("Combined")
print("reflection patches:", reflection_patches.shape)
print("other_patches:", other_patches.shape)


capetownsmall
city patch: (11, 1479, 2652)
city reflection patches: (146, 14, 56, 56)
city other patches: (1150, 14, 56, 56)
--------------------------------------------------
eldaein
city patch: (11, 1111, 943)
city reflection patches: (36, 14, 56, 56)
city other patches: (304, 14, 56, 56)
--------------------------------------------------
elgeneina
city patch: (11, 882, 899)
city reflection patches: (47, 14, 56, 56)
city other patches: (225, 14, 56, 56)
--------------------------------------------------
kianda
city patch: (11, 42, 34)
city reflection patches: (1, 14, 56, 56)
city other patches: (0, 14, 56, 56)
--------------------------------------------------
kibera
city patch: (11, 118, 256)
city reflection patches: (11, 14, 56, 56)
city other patches: (4, 14, 56, 56)
--------------------------------------------------
lower
city patch: (11, 44, 237)
city reflection patches: (5, 14, 56, 56)
city other patches: (0, 14, 56, 56)
--------------------------------------------------
makoko

# Train-val-test splits, ratio: 70:15:15

In [4]:
train_set, val_set, test_set = helper_functions.split_patches(reflection_patches, other_patches)
print("train:", train_set.shape)
print("val:", val_set.shape)
print("test:", test_set.shape)

train: (3154, 14, 56, 56)
val: (676, 14, 56, 56)
test: (676, 14, 56, 56)


# Save sets

In [ ]:
np.save(f"../data/deep_learning/train_patches.npy", train_set)
np.save(f"../data/deep_learning/val_patches.npy", val_set)
np.save(f"../data/deep_learning/test_patches.npy", test_set)

# Verify sets

In [7]:
rng = np.random.default_rng()
def get_10_random(arr): return [round(x.item(), 2) for x in rng.choice(arr.ravel(), size=10, replace=False)]

In [ ]:
train_set = np.load("../data/deep_learning/train_patches.npy")
x_train = train_set[:, :-1, :, :]
y_train = train_set[:, -1:, :, :]

print("train:")
print(f"x: {x_train.shape}, {x_train.dtype}, min: {x_train.min()}, max: {x_train.max()}")
print(f"random samples: {get_10_random(x_train)}")
print(f"y: {y_train.shape}, {y_train.dtype}, min: {y_train.min()}, max: {y_train.max()}")
print(f"random samples: {get_10_random(y_train)}")

assert np.all((x_train >= 0))
assert np.all((y_train == 0) | (y_train == 1))

del train_set, x_train, y_train
gc.collect()

train:
x: (3154, 13, 56, 56), float32, min: 0.0, max: 2.799999952316284
random samples: [0.16, 0.0, 0.25, 0.5, 0.12, 0.33, 0.14, 0.26, 0.0, 0.0]
y: (3154, 1, 56, 56), float32, min: 0.0, max: 1.0
random samples: [1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


247

In [ ]:
val_set = np.load("../data/deep_learning/val_patches.npy")
x_val = val_set[:, :-1, :, :]
y_val = val_set[:, -1:, :, :]

print("val:")
print(f"x: {x_val.shape}, {x_val.dtype}, min: {x_val.min()}, max: {x_val.max()}")
print(f"random samples: {get_10_random(x_val)}")
print(f"y: {y_val.shape}, {y_val.dtype}, min: {y_val.min()}, max: {y_val.max()}")
print(f"random samples: {get_10_random(y_val)}")

assert np.all((x_val >= 0))
assert np.all((y_val == 0) | (y_val == 1))

del val_set, x_val, y_val
gc.collect()

val:
x: (676, 13, 56, 56), float32, min: 0.0, max: 1.9365999698638916
random samples: [0.21, 0.01, 0.1, 0.05, 0.11, 0.0, 0.07, 0.2, 0.22, 0.23]
y: (676, 1, 56, 56), float32, min: 0.0, max: 1.0
random samples: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


11

In [ ]:
test_set = np.load("../data/deep_learning/test_patches.npy")
x_test = test_set[:, :-1, :, :]
y_test = test_set[:, -1:, :, :]

print("test:")
print(f"x: {x_test.shape}, {x_test.dtype}, min: {x_test.min()}, max: {x_test.max()}")
print(f"random samples: {get_10_random(x_test)}")
print(f"y: {y_test.shape}, {y_test.dtype}, min: {y_test.min()}, max: {y_test.max()}")
print(f"random samples: {get_10_random(y_test)}")

assert np.all((x_test >= 0))
assert np.all((y_test == 0) | (y_test == 1))

del test_set, x_test, y_test
gc.collect()

test:
x: (676, 13, 56, 56), float32, min: 0.0, max: 2.0638999938964844
random samples: [0.06, 0.13, 0.14, 0.12, 0.03, 0.0, 0.0, 0.16, 0.03, 0.16]
y: (676, 1, 56, 56), float32, min: 0.0, max: 1.0
random samples: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


11